# Wind Forecast Monitoring and Reliability Analysis

## Objective

This notebook analyzes UK national wind generation forecasts for January 2024 using the Elexon BMRS datasets.

The work has two goals:

1. **Forecast error analysis**  
   Understand the error characteristics of the forecast dataset, including:
   - overall error distribution
   - variation with forecast horizon
   - variation by time of day

2. **Wind reliability analysis**  
   Analyze historical actual wind generation and recommend how many MW of wind power can be **reliably expected** to be available to meet electricity demand.

## Scope

- Geography: United Kingdom national-level wind generation
- Period: January 2024
- Actuals dataset: `FUELHH` filtered to `fuelType = WIND`
- Forecast dataset: `WINDFOR`
- Forecast horizon range considered: `0–48 hours`

## Important modeling rule used in the app

For each target time and chosen forecast horizon `H`, the forecast used is:

- the **latest available forecast**
- whose `publishTime <= startTime - H`

This matches the assignment requirement:
> show the value of the latest forecast which was created at least H hours before each target time.

## Notes

During exploration, I found that:
- actual generation values are available every **30 minutes**
- forecast target times in the retrieved January 2024 data are available mostly at **hourly** timestamps

Because of that, forecast coverage is incomplete when aligning directly against all 30-minute actual timestamps. Missing forecasts are left missing rather than imputed.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_ROOT = DATA_ROOT / "raw"

ACTUALS_PATH = RAW_DATA_ROOT / "actuals_jan_2024.csv"
FORECASTS_PATH = RAW_DATA_ROOT / "forecasts_jan_2024.csv"

print("Actuals path exists:", ACTUALS_PATH.exists(), ACTUALS_PATH)
print("Forecasts path exists:", FORECASTS_PATH.exists(), FORECASTS_PATH)

In [ ]:
actuals = pd.read_csv(ACTUALS_PATH)
forecasts = pd.read_csv(FORECASTS_PATH)

print("Actuals shape:", actuals.shape)
print("Forecasts shape:", forecasts.shape)

In [ ]:
actuals.head()

In [ ]:
forecasts.head()

In [ ]:
actuals["startTime"] = pd.to_datetime(actuals["startTime"], utc=True)

if "publishTime" in actuals.columns:
    actuals["publishTime"] = pd.to_datetime(actuals["publishTime"], utc=True, errors="coerce")

forecasts["startTime"] = pd.to_datetime(forecasts["startTime"], utc=True)
forecasts["publishTime"] = pd.to_datetime(forecasts["publishTime"], utc=True)

if "horizon_hours" in forecasts.columns:
    forecasts["horizon_hours"] = forecasts["horizon_hours"].astype(float)
else:
    forecasts["horizon_hours"] = (
        forecasts["startTime"] - forecasts["publishTime"]
    ).dt.total_seconds() / 3600

## 1. Initial data quality checks

Before analyzing forecast errors, I first verify:
- dataset size
- column availability
- time range
- duplicates
- missing values
- timestamp cadence

In [ ]:
print("Actuals columns:", list(actuals.columns))
print("Forecasts columns:", list(forecasts.columns))

In [ ]:
summary_rows = [
    {
        "dataset": "actuals",
        "rows": len(actuals),
        "min_startTime": actuals["startTime"].min(),
        "max_startTime": actuals["startTime"].max(),
        "missing_startTime": actuals["startTime"].isna().sum(),
        "missing_generation": actuals["generation"].isna().sum(),
    },
    {
        "dataset": "forecasts",
        "rows": len(forecasts),
        "min_startTime": forecasts["startTime"].min(),
        "max_startTime": forecasts["startTime"].max(),
        "missing_startTime": forecasts["startTime"].isna().sum(),
        "missing_publishTime": forecasts["publishTime"].isna().sum(),
        "missing_generation": forecasts["generation"].isna().sum(),
    },
]

pd.DataFrame(summary_rows)

In [ ]:
actual_duplicates = actuals.duplicated(subset=["startTime"]).sum()
forecast_duplicates = forecasts.duplicated(subset=["startTime", "publishTime"]).sum()

print("Duplicate actual timestamps:", actual_duplicates)
print("Duplicate forecast (startTime, publishTime) pairs:", forecast_duplicates)

In [ ]:
actual_time_diffs = actuals["startTime"].sort_values().diff().dropna()
forecast_target_diffs = forecasts["startTime"].sort_values().diff().dropna()

print("Actual cadence counts:")
print(actual_time_diffs.value_counts().head(10))

print("\nForecast target-time cadence counts:")
print(forecast_target_diffs.value_counts().head(10))

In [ ]:
actual_minutes = sorted(actuals["startTime"].dt.minute.astype(str).str.zfill(2).unique())
forecast_minutes = sorted(forecasts["startTime"].dt.minute.astype(str).str.zfill(2).unique())

print("Unique minute values in actual target times:", actual_minutes)
print("Unique minute values in forecast target times:", forecast_minutes)

## Interpretation of data quality checks

Key observations from the raw data inspection:

1. **Actual generation series is half-hourly**  
   The actuals dataset provides target times at 30-minute resolution.

2. **Forecast target times are mostly hourly**  
   The retrieved forecast dataset for January 2024 appears to contain target timestamps mainly on the hour.

3. **Consequence for monitoring and analysis**  
   When matching actuals and forecasts by exact target timestamp, some actual half-hour timestamps do not have a corresponding forecast point. These cases should remain missing rather than being interpolated in the analytical dataset.

This is important because forecast coverage and error statistics should be interpreted only over timestamps where an eligible forecast actually exists.

In [ ]:
print("Actuals date range:")
print(actuals["startTime"].min(), "to", actuals["startTime"].max())

print("\nForecast publish time range:")
print(forecasts["publishTime"].min(), "to", forecasts["publishTime"].max())

print("\nForecast target time range:")
print(forecasts["startTime"].min(), "to", forecasts["startTime"].max())

print("\nForecast horizon summary:")
print(forecasts["horizon_hours"].describe())